In [ ]:
import os

# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd

In [ ]:
%%time
### cell 0 ###

factor = 1
data = pd.read_csv(
    "/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/week1.csv"
)
scout = pd.read_csv(
    "/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/pffScoutingData.csv"
)
plays = pd.read_csv(
    "/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/plays.csv"
)
players = pd.read_csv(
    "/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/players.csv"
)

# Let's merge these data into one set
data = data.merge(scout, how="left")
data = data.sample(frac=factor, random_state=0)
data.shape

In [ ]:
%%time
### cell 1 ###

data.info()

In [ ]:
%%time
### cell 2 ###

import numpy as np

# vectorized index calculation for ball_snap + next 5 frames (0–5)
start_idxs = data.index[data["event"] == "ball_snap"].to_numpy()
idxs = (start_idxs[:, None] + np.arange(6)).ravel()
# select those rows
df = data.reindex(idxs)

In [ ]:
%%time
### cell 3 ###

gid = 2021090900
pid = 97
nid = 25511
df.loc[(df["gameId"] == gid) & (df["playId"] == pid) & (df["nflId"] == nid)]

In [ ]:
%%time
### cell 4 ###

df = df.merge(
    data.query("team == 'football' and frameId == 1")[["gameId", "playId", "x"]].rename(
        columns={"x": "los"}
    ),
    on=["gameId", "playId"],
    how="inner",
)

In [ ]:
%%time
### cell 5 ###

df.loc[(df["gameId"] == gid) & (df["playId"] == pid) & (df["nflId"] == nid)]

In [ ]:
%%time
### cell 6 ###

# Compute signed distance from the line of scrimmage in one vectorized step
# (right plays get +1, left plays get -1)
df["los_diff"] = (df["x"] - df["los"]) * df["playDirection"].map(
    {"right": 1, "left": -1}
)

# Merge possessionTeam onto df in a single, keyed join
df = plays[["gameId", "playId", "possessionTeam"]].merge(df, on=["gameId", "playId"])

In [ ]:
%%time
### cell 7 ###

df.loc[(df["gameId"] == gid) & (df["playId"] == pid) & (df["nflId"] == nid)]

In [ ]:
%%time
### cell 8 ###

# vectorized assignment for posTeam in one step
df["posTeam"] = (
    df["possessionTeam"]
    .eq(df["team"])  # True if on possession team
    .astype("int8")  # 1 for True, 0 for False
    .mask(df["team"].eq("football"), -1)  # -1 for the football
)

# compute mean speed and acceleration per player directly on the subset
snap_speed = df.groupby("nflId", as_index=False)[["s", "a"]].mean()

In [ ]:
%%time
### cell 9 ###

snap_speed.head()

In [ ]:
%%time
### cell 10 ###

# Compute per-play los_diff based on posTeam using a single groupby
los_diff = (
    df.groupby(["gameId", "playId", "nflId"], as_index=False)
    .agg(
        posTeam=("posTeam", "first"),
        los_max=("los_diff", "max"),
        los_min=("los_diff", "min"),
    )
    .assign(los_diff=lambda x: x["los_max"].where(x["posTeam"] == 1, x["los_min"]))
    .groupby("nflId", as_index=False)
    .agg(los_diff=("los_diff", "mean"))
)

# Merge back and rename
snap_speed = snap_speed.merge(los_diff, on="nflId").rename(
    columns={"s": "snap_s", "a": "snap_a", "los_diff": "snap_los_diff"}
)

In [ ]:
%%time
### cell 11 ###

snap_speed.head()

In [ ]:
%%time
### cell 12 ###

df_plt = players.loc[:, ["nflId", "officialPosition", "displayName"]].merge(snap_speed)

In [ ]:
%%time
### cell 13 ###

df_plt.loc[
    (df_plt["officialPosition"] == "DE") & (df_plt["snap_los_diff"] < 0)
].sort_values("snap_los_diff")